# 📊 Monitorización y Observabilidad para Modelos ML

Este notebook demuestra la implementación de un sistema de monitorización proactiva para un modelo de Machine Learning en producción.

In [ ]:
# ============================================
# SECCIÓN 1: INSTALACIÓN DE DEPENDENCIAS
# ============================================
!pip install -q mlflow evidently scikit-learn pandas numpy matplotlib seaborn pyyaml
!pip install -q dataclasses-json joblib

print("✅ Dependencias instaladas correctamente")

In [ ]:
# ============================================
# SECCIÓN 2: CLONAR REPOSITORIO
# ============================================
import os
import sys

# Clonar repositorio
if not os.path.exists('ml-monitoring-observability-project'):
    !git clone https://github.com/tu-usuario/ml-monitoring-observability-project.git
    
%cd ml-monitoring-observability-project

# Agregar al path
sys.path.append(os.getcwd())

print("✅ Repositorio clonado")

In [ ]:
# ============================================
# SECCIÓN 3: IMPORTACIONES
# ============================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import mlflow
import mlflow.sklearn
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from evidently import ColumnMapping

print("✅ Librerías importadas")

In [ ]:
# ============================================
# SECCIÓN 4: CARGA DE DATOS
# ============================================
print("\n" + "="*50)
print("CARGANDO DATASET")
print("="*50)

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

print(f"Dataset shape: {X.shape}")
print(f"Features: {X.columns.tolist()}")

# Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

reference_data = X_train.copy()
reference_data['target'] = y_train.values

current_data = X_test.copy()
current_data['target'] = y_test.values

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

In [ ]:
# ============================================
# SECCIÓN 5: ENTRENAR MODELO
# ============================================
print("\n" + "="*50)
print("ENTRENANDO MODELO")
print("="*50)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluar
y_pred = model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Guardar modelos
import joblib
joblib.dump(model, 'model.joblib')
joblib.dump(scaler, 'scaler.joblib')

print("✅ Modelo guardado")

In [ ]:
# ============================================
# SECCIÓN 6: DETECCIÓN DE DATA DRIFT
# ============================================
print("\n" + "="*50)
print("DETECCIÓN DE DATA DRIFT")
print("="*50)

column_mapping = ColumnMapping()
column_mapping.target = 'target'
column_mapping.numerical_features = housing.feature_names

drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(
    reference_data=reference_data,
    current_data=current_data,
    column_mapping=column_mapping
)

drift_report.save_html('evidence/logs/data_drift_report.html')
print("✅ Reporte de drift guardado en evidence/logs/data_drift_report.html")

In [ ]:
# ============================================
# SECCIÓN 7: DASHBOARD OPERATIVO
# ============================================
print("\n" + "="*50)
print("GENERANDO DASHBOARD")
print("="*50)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Distribución de targets
axes[0, 0].hist(y_train, bins=30, alpha=0.7, label='Train')
axes[0, 0].hist(y_test, bins=30, alpha=0.7, label='Test')
axes[0, 0].set_title('Distribución de Target')
axes[0, 0].legend()

# 2. Predicciones vs Reales
axes[0, 1].scatter(y_test, y_pred, alpha=0.5)
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0, 1].set_title('Predicciones vs Reales')
axes[0, 1].set_xlabel('Real')
axes[0, 1].set_ylabel('Predicción')

# 3. Importancia de features
importance = pd.DataFrame({
    'feature': housing.feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
axes[1, 0].barh(importance['feature'], importance['importance'])
axes[1, 0].set_title('Importancia de Features')

# 4. Error distribution
errors = y_test - y_pred
axes[1, 1].hist(errors, bins=30, alpha=0.7)
axes[1, 1].set_title('Distribución de Errores')
axes[1, 1].set_xlabel('Error')

plt.tight_layout()
plt.savefig('evidence/logs/operational_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Dashboard guardado en evidence/logs/operational_dashboard.png")

In [ ]:
# ============================================
# SECCIÓN 8: SIMULAR INCIDENTES
# ============================================
print("\n" + "="*50)
print("SIMULANDO INCIDENTES")
print("="*50)

# Ejecutar script de simulación
!python scripts/simulate_incidents.py

# Mostrar resultados
with open('evidence/logs/runbook_executions.json', 'r') as f:
    results = json.load(f)

print("\n📊 Resultados de las simulaciones:")
for sim in results['simulations']:
    status = '✅' if sim['result']['status'] == 'completed' else '❌'
    print(f"{status} {sim['type']}: {sim['result']['status']} ({sim['result']['duration_seconds']:.1f}s)")

In [ ]:
# ============================================
# SECCIÓN 9: RESUMEN FINAL
# ============================================
print("\n" + "="*50)
print("📊 RESUMEN DEL PROYECTO")
print("="*50)

summary = {
    'project': 'Monitorización de Modelos ML',
    'dataset': 'California Housing',
    'model_type': 'RandomForestRegressor',
    'rmse': rmse,
    'r2': r2,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

with open('evidence/logs/monitoring_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\n✅ Proyecto completado exitosamente")